In [ ]:
# ===========================================
# 1. Mount Google Drive and define file paths
# ===========================================
from google.colab import drive
drive.mount('/content/drive')

# Define the location of your training file inside Google Drive
DATA_DIR = "/content/drive/My Drive/SubtaskA"
TRAIN_FILE = f"{DATA_DIR}/subtaskA_train_monolingual.jsonl"
OUTPUT_FILE = f"{DATA_DIR}/subtaskA_train_paraphrased_label0.jsonl"
PICKLE_FILE = f"{DATA_DIR}/real2para.pkl"  # For intermediate saving

Mounted at /content/drive


In [ ]:
# ===========================================
# 2. Install and import required libraries
# ===========================================
!pip install -q transformers jsonlines

import jsonlines
import pickle
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm
import os

MODEL_NAME = "ibm-research/qcpg-sentences"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [ ]:
# ===========================================
# 3. Load and extract human-written sentences
# ===========================================
def extract_sentences(text):
    text = text.replace("\n", " \n")
    sentences = []
    for line in text.split(". "):
        sentences.extend(line.split("\n"))
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    return sentences

data = []
sentence_pool = set()

with jsonlines.open(TRAIN_FILE) as reader:
    for item in reader:
        if item["label"] == 0:
            data.append(item)
            sentence_pool.update(extract_sentences(item["text"]))

sentence_list = sorted(list(sentence_pool), key=lambda x: len(x.split()), reverse=True)
print(f"Total unique sentences to paraphrase: {len(sentence_list)}")


Total unique sentences to paraphrase: 2349153


In [ ]:
# ===========================================
# 4. Load or create the paraphrase mapping (real2para)
# ===========================================
BATCH_SIZE = 256

if os.path.exists(PICKLE_FILE):
    print("Loading cached paraphrase results from pickle...")
    with open(PICKLE_FILE, "rb") as f:
        real2para = pickle.load(f)
else:
    print("Generating paraphrases from scratch...")
    real2para = {}

    def paraphrase_batch(batch_texts):
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = model.generate(inputs["input_ids"], max_new_tokens=83)
        return tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for i in tqdm(range(0, len(sentence_list), BATCH_SIZE)):
        batch = sentence_list[i:i+BATCH_SIZE]
        if all(s in real2para for s in batch):
            continue  # Skip already processed batch
        try:
            results = paraphrase_batch(batch)
            real2para.update(dict(zip(batch, results)))
            # Save progress after each batch
            with open(PICKLE_FILE, "wb") as f:
                pickle.dump(real2para, f)
        except Exception as e:
            print(f"Error in batch {i}: {e}")

    real2para[""] = ""


NameError: name 'os' is not defined

In [ ]:
# ===========================================
# 5. Reconstruct augmented text from mapping
# ===========================================
def reconstruct_text(original_text):
    reconstructed = []
    original_text = original_text.replace("\n", " \n")
    for line in original_text.split(". "):
        sub_lines = line.split("\n")
        new_lines = [real2para.get(s.strip(), s) for s in sub_lines]
        reconstructed.append("\n".join(new_lines))
    return ". ".join(reconstructed)


In [ ]:
# ===========================================
# 6. Apply augmentation to label=0 samples and save
# ===========================================
augmented_data = []

for item in tqdm(data):
    new_item = item.copy()
    new_item["gen_text"] = reconstruct_text(item["text"])
    augmented_data.append(new_item)

with jsonlines.open(OUTPUT_FILE, mode="w") as writer:
    writer.write_all(augmented_data)

print(f"Augmented data saved to: {OUTPUT_FILE}")


In [ ]:
from google.colab import files

files.download("train_paraphrased_label0.jsonl")
